# NeSLE A100 Benchmark + Native PPO

This notebook clones the GitHub repo, installs the Mario RL harness, builds NeSLE's CUDA extension for an A100, runs preflight/correctness checks, then launches the limits-first benchmark campaign.

Use a Colab **A100 80GB** runtime. Artifacts are written to Google Drive so they survive disconnects.

In [ ]:
#@title Campaign Config
REPO_URL = "https://github.com/hbofz/Nesle-codex.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PROJECT_SUBDIR = "project/mario-rl-ram"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/mario_rl"  #@param {type:"string"}
ROM_DRIVE_PATH = "/content/drive/MyDrive/mario_rl/roms/Super Mario Bros. (World).nes"  #@param {type:"string"}
RUN_NAME = "nesle_a100"  #@param {type:"string"}
CUDA_ARCH = "sm_80"  #@param ["sm_80", "sm_90"] {allow-input: true}
RUN_CORRECTNESS = True  #@param {type:"boolean"}
FULL_CAMPAIGN = True  #@param {type:"boolean"}

REPO_DIR = "/content/Nesle-codex"
PROJECT_DIR = f"{REPO_DIR}/{PROJECT_SUBDIR}"
OUTPUT_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
ROM_PROJECT_PATH = f"{PROJECT_DIR}/roms/Super Mario Bros. (World).nes"
SNAPSHOT_PATH = f"{REPO_DIR}/docs/data/smb_level1_1.state"

print("repo:", REPO_URL)
print("branch:", BRANCH)
print("project:", PROJECT_DIR)
print("output:", OUTPUT_DIR)

## 1. Mount Drive And Clone Repo

If the repo is private, add a Colab secret named `GITHUB_TOKEN` with read access before running this cell.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

drive.mount('/content/drive')

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with code {completed.returncode}: {cmd}")
    return completed

clone_url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if token and clone_url.startswith('https://github.com/'):
    clone_url = clone_url.replace('https://github.com/', f'https://{token}@github.com/', 1)

if Path(REPO_DIR).exists():
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', '--branch', BRANCH, clone_url, REPO_DIR])

print('repo sha:')
run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR)
print('project exists:', Path(PROJECT_DIR).is_dir())

## 2. Install, Build CUDA, And Copy ROM

The benchmark CLI can install/build NeSLE via `--setup`, but installing the harness first exposes `mario-nesle-bench` as a shell command.

In [ ]:
import sys

run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-e', PROJECT_DIR])

Path(f'{PROJECT_DIR}/roms').mkdir(parents=True, exist_ok=True)
if not Path(ROM_DRIVE_PATH).is_file():
    raise FileNotFoundError(f'ROM not found in Drive: {ROM_DRIVE_PATH}')
shutil.copy2(ROM_DRIVE_PATH, ROM_PROJECT_PATH)
print('ROM copied to:', ROM_PROJECT_PATH)

print('GPU:')
run('nvidia-smi', check=False)

print('PyTorch CUDA:')
run([sys.executable, '-c', 'import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")'])

## 3. Preflight

This builds NeSLE `_cuda_core`, imports it, loads the snapshot, checks gameplay RAM, records hardware metadata, and optionally runs the falsifiability correctness benchmark.

In [ ]:
preflight_cmd = [
    'mario-nesle-bench', 'preflight',
    '--setup',
    '--cuda-arch', CUDA_ARCH,
    '--nesle-root', REPO_DIR,
    '--rom', ROM_PROJECT_PATH,
    '--snapshot', SNAPSHOT_PATH,
    '--output-dir', OUTPUT_DIR,
]
if RUN_CORRECTNESS:
    preflight_cmd.append('--run-correctness')
run(preflight_cmd, cwd=PROJECT_DIR)

## 4. Env-Only Throughput Sweep

This measures raw cuda-console environment throughput with snapshot reset, no render, and no observation copy.

In [ ]:
#@title Env Sweep Options
ENV_COUNTS = "1,8,32,128,512,1024,2048,4096,8192,16384,32768,65536,98304,131072"  #@param {type:"string"}
ENV_WARMUP_STEPS = 30  #@param {type:"integer"}
ENV_TIMED_STEPS = 200  #@param {type:"integer"}

env_cmd = [
    'mario-nesle-bench', 'env-sweep',
    '--nesle-root', REPO_DIR,
    '--rom', ROM_PROJECT_PATH,
    '--snapshot', SNAPSHOT_PATH,
    '--output-dir', OUTPUT_DIR,
    '--env-counts', ENV_COUNTS,
    '--warmup-steps', str(ENV_WARMUP_STEPS),
    '--timed-steps', str(ENV_TIMED_STEPS),
]
run(env_cmd, cwd=PROJECT_DIR)

## 5. Native PPO Short Sweep

This runs short GPU-resident PPO jobs. Use this to identify which env counts are stable before launching the long run.

In [ ]:
#@title PPO Sweep Options
PPO_COUNTS = "1024,4096,8192,16384,32768,65536,98304,131072"  #@param {type:"string"}
PPO_UPDATES = 2  #@param {type:"integer"}
N_STEPS = 128  #@param {type:"integer"}
BATCH_SIZE = 8192  #@param {type:"integer"}
HIDDEN_SIZE = 256  #@param {type:"integer"}

ppo_cmd = [
    'mario-nesle-bench', 'ppo-sweep',
    '--nesle-root', REPO_DIR,
    '--rom', ROM_PROJECT_PATH,
    '--snapshot', SNAPSHOT_PATH,
    '--output-dir', OUTPUT_DIR,
    '--ppo-env-counts', PPO_COUNTS,
    '--ppo-updates', str(PPO_UPDATES),
    '--n-steps', str(N_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-size', str(HIDDEN_SIZE),
    '--ppo-action-space', 'mario',
]
run(ppo_cmd, cwd=PROJECT_DIR)

## 6. Full Campaign Or Long Stress Run

Run this after the short sweeps look healthy. If `FULL_CAMPAIGN` is enabled, this runs the combined command. Otherwise it runs a single 75M-timestep stress job.

In [ ]:
#@title Stress Options
STRESS_ENVS = 65536  #@param {type:"integer"}
STRESS_TIMESTEPS = 75000000  #@param {type:"integer"}

if FULL_CAMPAIGN:
    cmd = [
        'mario-nesle-bench', 'all',
        '--nesle-root', REPO_DIR,
        '--rom', ROM_PROJECT_PATH,
        '--snapshot', SNAPSHOT_PATH,
        '--output-dir', OUTPUT_DIR,
        '--env-counts', ENV_COUNTS,
        '--ppo-env-counts', PPO_COUNTS,
        '--ppo-action-space', 'mario',
        '--n-steps', str(N_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--hidden-size', str(HIDDEN_SIZE),
    ]
else:
    cmd = [
        'mario-nesle-bench', 'stress',
        '--nesle-root', REPO_DIR,
        '--rom', ROM_PROJECT_PATH,
        '--snapshot', SNAPSHOT_PATH,
        '--output-dir', OUTPUT_DIR,
        '--stress-envs', str(STRESS_ENVS),
        '--stress-timesteps', str(STRESS_TIMESTEPS),
        '--ppo-action-space', 'mario',
        '--n-steps', str(N_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--hidden-size', str(HIDDEN_SIZE),
    ]
run(cmd, cwd=PROJECT_DIR)

## 7. Inspect Outputs

Open the generated Markdown report and list all artifacts written to Drive.

In [ ]:
from IPython.display import Markdown, display

report = Path(OUTPUT_DIR) / 'nesle_a100_report.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('No report yet:', report)

print('\nArtifacts:')
for path in sorted(Path(OUTPUT_DIR).rglob('*')):
    if path.is_file():
        print(path)